In [3]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
!pip install -q scanpy anndata scikit-learn torch torchvision torchaudio
!pip install -q scgpt
!pip uninstall -y torchtext scgpt
!pip install -q torch==2.3.1 torchtext==0.18.0
!pip install -q scgpt scanpy anndata scikit-learn

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 3.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.2/91.2 kB 10.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 41.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 174.3/174.3 kB 18.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.1/60.1 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 102.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 295.7/295.7 kB 26.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 102.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 2.3.3 which is incompatible.
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━

In [13]:
import os
import json
import numpy as np
import pandas as pd
import scanpy as sc

PRETRAINED_DIR = "/content/drive/MyDrive/scGPT_human"
VAL_DIR = "/content/drive/MyDrive/CompGenomicsValidation"
VAL_PATH = f"{VAL_DIR}/validation_scGPT_input.h5ad"
PDAC_PATH = "/content/drive/MyDrive/PDAC_ADJ_Final_Annotated.h5ad"

RAW_CACHE = f"{VAL_DIR}/raw_scgpt_cache"
os.makedirs(RAW_CACHE, exist_ok=True)

print("Pretrained:", PRETRAINED_DIR)
print("Validation:", VAL_PATH)
print("PDAC:", PDAC_PATH)

Pretrained: /content/drive/MyDrive/scGPT_human
Validation: /content/drive/MyDrive/CompGenomicsValidation/validation_scGPT_input.h5ad
PDAC: /content/drive/MyDrive/PDAC_ADJ_Final_Annotated.h5ad


In [14]:
from collections import OrderedDict
from torchtext.vocab import vocab as torchtext_vocab
from scgpt.tokenizer.gene_tokenizer import GeneVocab

vocab_path = f"{PRETRAINED_DIR}/vocab.json"

with open(vocab_path, "r") as f:
    token2idx = json.load(f)

@classmethod
def fixed_from_dict(cls, token2idx, default_token=None):
    ordered = OrderedDict(sorted(token2idx.items(), key=lambda x: x[1]))
    ordered_freq = OrderedDict((tok, 1) for tok in ordered.keys())

    obj = cls([])
    obj.vocab = torchtext_vocab(ordered_freq)

    if default_token is not None:
        obj.set_default_token(default_token)

    return obj

GeneVocab.from_dict = fixed_from_dict

vocab = GeneVocab.from_file(vocab_path)
print("Loaded vocab size:", len(vocab))

Loaded vocab size: 60697


In [15]:
adata_val = sc.read_h5ad(VAL_PATH)
adata_val.var_names_make_unique()
adata_val.var["gene_name"] = adata_val.var_names.astype(str)

TRUE_COL = "cell_type_label"

print(adata_val)
print("Validation labels:")
print(adata_val.obs[TRUE_COL].value_counts())

AnnData object with n_obs × n_vars = 28066 × 3000
    obs: 'sample_id', 'condition', 'tissue_type', 'replicate', 'batch', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_50_genes', 'pct_counts_in_top_100_genes', 'pct_counts_in_top_200_genes', 'pct_counts_in_top_500_genes', 'total_counts_mt', 'log1p_total_counts_mt', 'pct_counts_mt', 'total_counts_ribo', 'log1p_total_counts_ribo', 'pct_counts_ribo', 'leiden_0.3', 'leiden_0.5', 'leiden_0.9', 'pdac_annotation', 'adj_annotation', 'healthy_annotation', 'cell_type_label'
    var: 'mt', 'ribo', 'n_cells_by_counts', 'mean_counts', 'log1p_mean_counts', 'pct_dropout_by_counts', 'total_counts', 'log1p_total_counts', 'n_cells', 'highly_variable', 'means', 'dispersions', 'dispersions_norm', 'highly_variable_nbatches', 'highly_variable_intersection', 'gene_name'
    uns: 'adj_annotation_colors', 'cell_type_label_colors', 'condition_colors', 'hvg', 'leiden_0.3', 'leiden_0.3_colors', 'leiden_0.5

In [16]:
adata = sc.read_h5ad(PDAC_PATH)
adata.var_names_make_unique()
adata.var["gene_name"] = adata.var_names.astype(str)

TEST_SAMPLES = ["ADJ4", "ADJ5", "PDAC1"]
TRAIN_SAMPLES = [s for s in adata.obs["sample_id"].unique() if s not in TEST_SAMPLES]

adata_train = adata[adata.obs["sample_id"].isin(TRAIN_SAMPLES)].copy()
adata_test = adata[adata.obs["sample_id"].isin(TEST_SAMPLES)].copy()

label_col = "scGPT_target_label"

y_train = adata_train.obs[label_col].astype(str).values
y_test = adata_test.obs[label_col].astype(str).values

print("Train:", adata_train.shape)
print("Test:", adata_test.shape)
print(pd.Series(y_train).value_counts())

Train: (52481, 36601)
Test: (18593, 36601)
T-Cells        17253
Fibroblasts     7689
Ductal          7376
Macrophages     4680
Acinar          4438
B-Cells         4172
Neutrophils     2187
Endothelial     1793
Stellate        1105
Plasma           557
Mast             479
NK               377
Endocrine        251
Schwann          124
Name: count, dtype: int64


In [17]:
from scgpt.tasks import embed_data

train_raw_path = f"{RAW_CACHE}/X_train_raw.npy"
test_raw_path = f"{RAW_CACHE}/X_test_raw.npy"
val_raw_path = f"{RAW_CACHE}/X_val_raw.npy"

if os.path.exists(train_raw_path) and os.path.exists(test_raw_path) and os.path.exists(val_raw_path):
    print("Loading cached raw embeddings...")
    X_train_raw = np.load(train_raw_path)
    X_test_raw = np.load(test_raw_path)
    X_val_raw = np.load(val_raw_path)

else:
    print("Embedding train with raw scGPT...")
    adata_train_raw = embed_data(
        adata_train,
        model_dir=PRETRAINED_DIR,
        gene_col="gene_name",
        batch_size=64,
        return_new_adata=True,
    )
    X_train_raw = adata_train_raw.X
    np.save(train_raw_path, X_train_raw)

    print("Embedding test with raw scGPT...")
    adata_test_raw = embed_data(
        adata_test,
        model_dir=PRETRAINED_DIR,
        gene_col="gene_name",
        batch_size=64,
        return_new_adata=True,
    )
    X_test_raw = adata_test_raw.X
    np.save(test_raw_path, X_test_raw)

    print("Embedding validation with raw scGPT...")
    adata_val_raw = embed_data(
        adata_val,
        model_dir=PRETRAINED_DIR,
        gene_col="gene_name",
        batch_size=64,
        return_new_adata=True,
    )
    X_val_raw = adata_val_raw.X
    np.save(val_raw_path, X_val_raw)

print("X_train_raw:", X_train_raw.shape)
print("X_test_raw:", X_test_raw.shape)
print("X_val_raw:", X_val_raw.shape)

Embedding train with raw scGPT...
scGPT - INFO - match 24285/36601 genes in vocabulary of size 60697.


/usr/local/lib/python3.12/dist-packages/scgpt/model/model.py:77: UserWarning: flash-attn is not installed, using pytorch transformer instead. Set use_fast_transformer=False to avoid this warning. Installing flash-attn is highly recommended.
  warnings.warn(
Embedding cells: 100%|██████████| 821/821 [08:45<00:00,  1.56it/s]
/usr/local/lib/python3.12/dist-packages/legacy_api_wrap/__init__.py:88: FutureWarning: The dtype argument is deprecated and will be removed in late 2024.
  return fn(*args_all, **kw)


Embedding test with raw scGPT...
scGPT - INFO - match 24285/36601 genes in vocabulary of size 60697.


/usr/local/lib/python3.12/dist-packages/scgpt/model/model.py:77: UserWarning: flash-attn is not installed, using pytorch transformer instead. Set use_fast_transformer=False to avoid this warning. Installing flash-attn is highly recommended.
  warnings.warn(
Embedding cells: 100%|██████████| 291/291 [03:05<00:00,  1.57it/s]
/usr/local/lib/python3.12/dist-packages/legacy_api_wrap/__init__.py:88: FutureWarning: The dtype argument is deprecated and will be removed in late 2024.
  return fn(*args_all, **kw)


Embedding validation with raw scGPT...
scGPT - INFO - match 2773/3000 genes in vocabulary of size 60697.


/usr/local/lib/python3.12/dist-packages/scgpt/model/model.py:77: UserWarning: flash-attn is not installed, using pytorch transformer instead. Set use_fast_transformer=False to avoid this warning. Installing flash-attn is highly recommended.
  warnings.warn(
Embedding cells: 100%|██████████| 439/439 [03:08<00:00,  2.32it/s]


X_train_raw: (52481, 512)
X_test_raw: (18593, 512)
X_val_raw: (28066, 512)


/usr/local/lib/python3.12/dist-packages/legacy_api_wrap/__init__.py:88: FutureWarning: The dtype argument is deprecated and will be removed in late 2024.
  return fn(*args_all, **kw)


In [18]:
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report
)

def metric_row(name, y_true, y_pred):
    return {
        "model": name,
        "accuracy": accuracy_score(y_true, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "precision_macro": precision_score(y_true, y_pred, average="macro", zero_division=0),
        "recall_macro": recall_score(y_true, y_pred, average="macro", zero_division=0),
        "f1_macro": f1_score(y_true, y_pred, average="macro", zero_division=0),
    }

test_results = []

lr_raw = LogisticRegression(
    max_iter=1000,
    C=1.0,
    solver="lbfgs",
    random_state=42,
    n_jobs=-1
)
lr_raw.fit(X_train_raw, y_train)

knn_raw = KNeighborsClassifier(
    n_neighbors=10,
    metric="euclidean",
    n_jobs=-1
)
knn_raw.fit(X_train_raw, y_train)

y_pred_lr_test = lr_raw.predict(X_test_raw)
y_pred_knn_test = knn_raw.predict(X_test_raw)

test_results.append(metric_row("Raw scGPT + LR", y_test, y_pred_lr_test))
test_results.append(metric_row("Raw scGPT + KNN k=10", y_test, y_pred_knn_test))

print("Raw scGPT + LR on original held-out test")
print(classification_report(y_test, y_pred_lr_test, zero_division=0))

print("Raw scGPT + KNN on original held-out test")
print(classification_report(y_test, y_pred_knn_test, zero_division=0))

adata_val.obs["pred_raw_lr"] = lr_raw.predict(X_val_raw)
adata_val.obs["pred_raw_knn"] = knn_raw.predict(X_val_raw)

pd.DataFrame(test_results)

Raw scGPT + LR on original held-out test
              precision    recall  f1-score   support

      Acinar       0.94      0.90      0.92      3877
     B-Cells       0.98      0.99      0.99       239
      Ductal       0.91      0.96      0.93      3742
   Endocrine       0.43      0.06      0.11       156
 Endothelial       0.99      0.98      0.98      1454
 Fibroblasts       0.95      0.97      0.96      2361
 Macrophages       0.95      0.96      0.96      1514
        Mast       1.00      0.94      0.97       510
          NK       0.92      0.79      0.85        28
 Neutrophils       0.96      0.98      0.97       181
      Plasma       0.78      0.87      0.82       151
     Schwann       0.80      0.76      0.78        63
    Stellate       0.94      0.92      0.93       462
     T-Cells       0.99      0.99      0.99      3855

    accuracy                           0.95     18593
   macro avg       0.90      0.86      0.87     18593
weighted avg       0.94      0.95      

,model,accuracy,balanced_accuracy,precision_macro,recall_macro,f1_macro
0,Raw scGPT + LR,0.947615,0.863639,0.895429,0.863639,0.868932
1,Raw scGPT + KNN k=10,0.960738,0.911305,0.912168,0.911305,0.910690


In [19]:
EPOCHS   = 60
LR       = 1e-3
PATIENCE = 10

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import LabelEncoder

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

le_raw = LabelEncoder()
y_train_enc = le_raw.fit_transform(y_train)
y_test_enc = le_raw.transform(y_test)

X_tr = torch.tensor(X_train_raw, dtype=torch.float32)
y_tr = torch.tensor(y_train_enc, dtype=torch.long)
X_te = torch.tensor(X_test_raw, dtype=torch.float32)
y_te = torch.tensor(y_test_enc, dtype=torch.long)

train_dl = DataLoader(
    TensorDataset(X_tr, y_tr),
    batch_size=512,
    shuffle=True,
    pin_memory=(device == "cuda")
)

class MLP(nn.Module):
    def __init__(self, in_dim, hidden_dims, n_classes, dropout=0.3):
        super().__init__()
        layers, prev = [], in_dim
        for h in hidden_dims:
            layers += [
                nn.Linear(prev, h),
                nn.BatchNorm1d(h),
                nn.ReLU(),
                nn.Dropout(dropout)
            ]
            prev = h
        layers.append(nn.Linear(prev, n_classes))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)

mlp_raw = MLP(
    in_dim=X_train_raw.shape[1],
    hidden_dims=[256, 128],
    n_classes=len(le_raw.classes_),
    dropout=0.3
).to(device)

optimizer = torch.optim.Adam(mlp_raw.parameters(), lr=LR, weight_decay=1e-4)
criterion = nn.CrossEntropyLoss()
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="max", factor=0.5, patience=5
)

best_val_acc, patience_counter = 0.0, 0
best_path = f"{RAW_CACHE}/best_raw_mlp.pt"

for epoch in range(1, EPOCHS + 1):
    mlp_raw.train()
    epoch_loss = 0.0

    for xb, yb in train_dl:
        xb, yb = xb.to(device), yb.to(device)

        optimizer.zero_grad()
        loss = criterion(mlp_raw(xb), yb)
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item() * len(xb)

    mlp_raw.eval()
    with torch.no_grad():
        val_pred = mlp_raw(X_te.to(device)).argmax(1).cpu()
        val_acc = (val_pred == y_te).float().mean().item()

    avg_loss = epoch_loss / len(X_tr)
    scheduler.step(val_acc)

    if epoch % 5 == 0 or epoch == 1:
        print(f"Epoch {epoch:3d}/{EPOCHS}  loss={avg_loss:.4f}  val_acc={val_acc:.4f}")

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        patience_counter = 0
        torch.save(mlp_raw.state_dict(), best_path)
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print(f"Early stopping at epoch {epoch}  (best={best_val_acc:.4f})")
            break

print(f"Best raw MLP held-out test accuracy: {best_val_acc:.4f}")

mlp_raw.load_state_dict(torch.load(best_path, map_location=device))
mlp_raw.to(device)
mlp_raw.eval()

with torch.no_grad():
    pred_test_enc = mlp_raw(torch.tensor(X_test_raw, dtype=torch.float32).to(device)).argmax(1).cpu().numpy()
    y_pred_mlp_test = le_raw.inverse_transform(pred_test_enc)

test_results.append(metric_row("Raw scGPT + MLP", y_test, y_pred_mlp_test))

print("Raw scGPT + MLP on original held-out test")
print(classification_report(y_test, y_pred_mlp_test, zero_division=0))

with torch.no_grad():
    pred_val_enc = mlp_raw(torch.tensor(X_val_raw, dtype=torch.float32).to(device)).argmax(1).cpu().numpy()

adata_val.obs["pred_raw_mlp"] = le_raw.inverse_transform(pred_val_enc)

test_results_df = pd.DataFrame(test_results).sort_values("f1_macro", ascending=False)
test_results_df

Device: cuda
Epoch   1/60  loss=0.3334  val_acc=0.9473
Epoch   5/60  loss=0.0534  val_acc=0.9656
Epoch  10/60  loss=0.0408  val_acc=0.9620
Epoch  15/60  loss=0.0352  val_acc=0.9632
Epoch  20/60  loss=0.0306  val_acc=0.9629
Epoch  25/60  loss=0.0219  val_acc=0.9654
Early stopping at epoch 27  (best=0.9697)
Best raw MLP held-out test accuracy: 0.9697
Raw scGPT + MLP on original held-out test
              precision    recall  f1-score   support

      Acinar       0.97      0.94      0.96      3877
     B-Cells       0.99      0.99      0.99       239
      Ductal       0.96      0.97      0.96      3742
   Endocrine       0.77      0.76      0.76       156
 Endothelial       0.98      1.00      0.99      1454
 Fibroblasts       0.96      0.98      0.97      2361
 Macrophages       0.97      0.98      0.98      1514
        Mast       0.99      0.99      0.99       510
          NK       0.85      0.82      0.84        28
 Neutrophils       0.98      0.99      0.99       181
      Plasma

,model,accuracy,balanced_accuracy,precision_macro,recall_macro,f1_macro
2,Raw scGPT + MLP,0.969720,0.926115,0.937463,0.926115,0.931375
1,Raw scGPT + KNN k=10,0.960738,0.911305,0.912168,0.911305,0.910690
0,Raw scGPT + LR,0.947615,0.863639,0.895429,0.863639,0.868932


In [20]:
TRUE_COL = "cell_type_label"

val_results = []

pred_cols = {
    "Raw scGPT + LR": "pred_raw_lr",
    "Raw scGPT + KNN k=10": "pred_raw_knn",
    "Raw scGPT + MLP": "pred_raw_mlp",
}

y_true_val = adata_val.obs[TRUE_COL].astype(str).values

for model_name, pred_col in pred_cols.items():
    y_pred_val = adata_val.obs[pred_col].astype(str).values

    val_results.append(metric_row(model_name, y_true_val, y_pred_val))

    print("\n" + "="*80)
    print(model_name, "on validation set")
    print("="*80)
    print(classification_report(y_true_val, y_pred_val, zero_division=0))

val_results_df = pd.DataFrame(val_results).sort_values("f1_macro", ascending=False)
val_results_df

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")



Raw scGPT + LR on validation set
              precision    recall  f1-score   support

      Acinar       0.97      0.94      0.95      1450
     B-Cells       0.98      0.82      0.89       962
      Ductal       0.97      0.98      0.97      2775
   Endocrine       0.00      0.00      0.00         0
 Endothelial       1.00      0.98      0.99       255
 Fibroblasts       0.90      0.97      0.93       257
 Macrophages       0.33      0.98      0.49      1533
        Mast       0.99      0.93      0.96       225
          NK       0.62      0.83      0.71      1296
 Neutrophils       0.94      0.42      0.58      5054
      Plasma       0.00      0.00      0.00         0
     Schwann       0.00      0.00      0.00         0
    Stellate       1.00      0.83      0.91       534
     T-Cells       0.98      0.94      0.96     13725

    accuracy                           0.84     28066
   macro avg       0.69      0.69      0.67     28066
weighted avg       0.92      0.84      0.85   

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")



Raw scGPT + KNN k=10 on validation set
              precision    recall  f1-score   support

      Acinar       0.89      0.94      0.91      1450
     B-Cells       0.99      0.81      0.89       962
      Ductal       0.97      0.92      0.94      2775
   Endocrine       0.00      0.00      0.00         0
 Endothelial       0.97      0.98      0.97       255
 Fibroblasts       0.57      0.89      0.69       257
 Macrophages       0.34      0.97      0.50      1533
        Mast       0.99      0.94      0.96       225
          NK       0.53      0.91      0.67      1296
 Neutrophils       0.93      0.44      0.59      5054
      Plasma       0.00      0.00      0.00         0
     Schwann       0.00      0.00      0.00         0
    Stellate       0.99      0.83      0.91       534
     T-Cells       0.99      0.90      0.94     13725

    accuracy                           0.82     28066
   macro avg       0.65      0.68      0.64     28066
weighted avg       0.91      0.82      0

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")



Raw scGPT + MLP on validation set
              precision    recall  f1-score   support

      Acinar       0.97      0.93      0.95      1450
     B-Cells       0.98      0.82      0.89       962
      Ductal       0.96      0.95      0.96      2775
   Endocrine       0.00      0.00      0.00         0
 Endothelial       0.97      1.00      0.98       255
 Fibroblasts       0.79      0.96      0.87       257
 Macrophages       0.32      0.99      0.48      1533
        Mast       0.99      0.96      0.97       225
          NK       0.46      0.90      0.61      1296
 Neutrophils       0.91      0.40      0.56      5054
      Plasma       0.00      0.00      0.00         0
     Schwann       0.00      0.00      0.00         0
    Stellate       1.00      0.83      0.91       534
     T-Cells       0.99      0.88      0.93     13725

    accuracy                           0.81     28066
   macro avg       0.67      0.69      0.65     28066
weighted avg       0.91      0.81      0.83  

,model,accuracy,balanced_accuracy,precision_macro,recall_macro,f1_macro
0,Raw scGPT + LR,0.842977,0.875166,0.691028,0.687630,0.668452
2,Raw scGPT + MLP,0.808487,0.875238,0.667314,0.687687,0.650807
1,Raw scGPT + KNN k=10,0.823630,0.866736,0.652824,0.681007,0.641905


In [21]:
OUT_H5AD = f"{VAL_DIR}/validation_with_raw_scgpt_predictions_FIXED.h5ad"
OUT_OBS = f"{VAL_DIR}/validation_raw_scgpt_predictions_FIXED_obs.csv"
OUT_VAL_METRICS = f"{VAL_DIR}/validation_raw_scgpt_metrics_FIXED.csv"
OUT_TEST_METRICS = f"{VAL_DIR}/heldout_test_raw_scgpt_metrics_FIXED.csv"

adata_val.write_h5ad(OUT_H5AD)
adata_val.obs.to_csv(OUT_OBS)
val_results_df.to_csv(OUT_VAL_METRICS, index=False)
test_results_df.to_csv(OUT_TEST_METRICS, index=False)

print("Saved:", OUT_H5AD)
print("Saved:", OUT_OBS)
print("Saved:", OUT_VAL_METRICS)
print("Saved:", OUT_TEST_METRICS)

adata_val.obs[["cell_type_label", "pred_raw_lr", "pred_raw_knn", "pred_raw_mlp"]].head()

Saved: /content/drive/MyDrive/CompGenomicsValidation/validation_with_raw_scgpt_predictions_FIXED.h5ad
Saved: /content/drive/MyDrive/CompGenomicsValidation/validation_raw_scgpt_predictions_FIXED_obs.csv
Saved: /content/drive/MyDrive/CompGenomicsValidation/validation_raw_scgpt_metrics_FIXED.csv
Saved: /content/drive/MyDrive/CompGenomicsValidation/heldout_test_raw_scgpt_metrics_FIXED.csv


,cell_type_label,pred_raw_lr,pred_raw_knn,pred_raw_mlp
AAACCCAGTCCGACGT-1_AdjNorm_TISSUE_1,Stellate,Stellate,Stellate,Stellate
AAACGAAAGCGACCCT-1_AdjNorm_TISSUE_1,Acinar,Acinar,Acinar,Acinar
AAACGAAAGCGGACAT-1_AdjNorm_TISSUE_1,T-Cells,T-Cells,T-Cells,T-Cells
AAACGAAAGTCACACT-1_AdjNorm_TISSUE_1,Acinar,Acinar,Acinar,Acinar
AAACGAACAACAGATA-1_AdjNorm_TISSUE_1,Macrophages,Macrophages,Macrophages,Macrophages


#FINE TUNING EMBEDDINGS AND PREDICTION ON VALIDATION SET

In [26]:
import os, glob, shutil

DRIVE_FINETUNED_ROOT = "/content/drive/MyDrive/scGPT_human/finetuned_pdac"
PRETRAINED_DIR = "/content/drive/MyDrive/scGPT_human"

ckpt_dirs = sorted(
    glob.glob(f"{DRIVE_FINETUNED_ROOT}/dev_*"),
    key=os.path.getmtime
)

assert ckpt_dirs, f"No finetuned runs found in {DRIVE_FINETUNED_ROOT}"

BEST_MODEL_DIR = ckpt_dirs[-1]

for fname in ["args.json", "vocab.json"]:
    src = f"{PRETRAINED_DIR}/{fname}"
    dst = f"{BEST_MODEL_DIR}/{fname}"
    if not os.path.exists(dst):
        print(f"Copying {fname} into checkpoint dir")
        shutil.copy(src, dst)

BEST_MODEL_PATH = f"{BEST_MODEL_DIR}/best_model.pt"

assert os.path.exists(BEST_MODEL_PATH), f"Missing: {BEST_MODEL_PATH}"
assert os.path.exists(f"{BEST_MODEL_DIR}/vocab.json"), "Missing vocab.json"
assert os.path.exists(f"{BEST_MODEL_DIR}/args.json"), "Missing args.json"

print("Using BEST_MODEL_DIR:", BEST_MODEL_DIR)
print("Files:", os.listdir(BEST_MODEL_DIR))

Using BEST_MODEL_DIR: /content/drive/MyDrive/scGPT_human/finetuned_pdac/dev_eyeGPT-Apr20-18-31-31
Files: ['id2type.json', 'dev_train_args.yml', 'vocab.json', 'protocol_finetune.py', 'model_e1.pt', 'model_e3.pt', 'model_e5.pt', 'model_e6.pt', 'model_e7.pt', 'run.log', 'best_model.pt', 'args.json']


In [27]:
import os, glob, shutil

DRIVE_FINETUNED_ROOT = "/content/drive/MyDrive/scGPT_human/finetuned_pdac"
PRETRAINED_DIR = "/content/drive/MyDrive/scGPT_human"

ckpt_dirs = sorted(
    glob.glob(f"{DRIVE_FINETUNED_ROOT}/dev_*"),
    key=os.path.getmtime
)

assert ckpt_dirs, f"No finetuned runs found in {DRIVE_FINETUNED_ROOT}"

BEST_MODEL_DIR = ckpt_dirs[-1]

for fname in ["args.json", "vocab.json"]:
    src = f"{PRETRAINED_DIR}/{fname}"
    dst = f"{BEST_MODEL_DIR}/{fname}"
    if not os.path.exists(dst):
        print(f"Copying {fname} into checkpoint dir")
        shutil.copy(src, dst)

BEST_MODEL_PATH = f"{BEST_MODEL_DIR}/best_model.pt"

assert os.path.exists(BEST_MODEL_PATH), f"Missing: {BEST_MODEL_PATH}"
assert os.path.exists(f"{BEST_MODEL_DIR}/vocab.json"), "Missing vocab.json"
assert os.path.exists(f"{BEST_MODEL_DIR}/args.json"), "Missing args.json"

print("Using BEST_MODEL_DIR:", BEST_MODEL_DIR)
print("Files:", os.listdir(BEST_MODEL_DIR))

Using BEST_MODEL_DIR: /content/drive/MyDrive/scGPT_human/finetuned_pdac/dev_eyeGPT-Apr20-18-31-31
Files: ['id2type.json', 'dev_train_args.yml', 'vocab.json', 'protocol_finetune.py', 'model_e1.pt', 'model_e3.pt', 'model_e5.pt', 'model_e6.pt', 'model_e7.pt', 'run.log', 'best_model.pt', 'args.json']


In [28]:
from scgpt.tasks import embed_data
import numpy as np
import os

FT_CACHE = f"{VAL_DIR}/finetuned_scgpt_cache"
os.makedirs(FT_CACHE, exist_ok=True)

train_ft_path = f"{FT_CACHE}/X_train_ft.npy"
test_ft_path  = f"{FT_CACHE}/X_test_ft.npy"
val_ft_path   = f"{FT_CACHE}/X_val_ft.npy"

if os.path.exists(train_ft_path) and os.path.exists(test_ft_path) and os.path.exists(val_ft_path):
    print("Loading cached fine-tuned embeddings...")
    X_train_ft = np.load(train_ft_path)
    X_test_ft  = np.load(test_ft_path)
    X_val_ft   = np.load(val_ft_path)

else:
    print("Embedding train with fine-tuned scGPT...")
    X_train_ft = embed_data(
        adata_train,
        model_dir=BEST_MODEL_DIR,
        gene_col="gene_name",
        batch_size=64,
        return_new_adata=True,
    ).X
    np.save(train_ft_path, X_train_ft)

    print("Embedding test with fine-tuned scGPT...")
    X_test_ft = embed_data(
        adata_test,
        model_dir=BEST_MODEL_DIR,
        gene_col="gene_name",
        batch_size=64,
        return_new_adata=True,
    ).X
    np.save(test_ft_path, X_test_ft)

    print("Embedding validation with fine-tuned scGPT...")
    X_val_ft = embed_data(
        adata_val,
        model_dir=BEST_MODEL_DIR,
        gene_col="gene_name",
        batch_size=64,
        return_new_adata=True,
    ).X
    np.save(val_ft_path, X_val_ft)

print("X_train_ft:", X_train_ft.shape)
print("X_test_ft:", X_test_ft.shape)
print("X_val_ft:", X_val_ft.shape)

Embedding train with fine-tuned scGPT...
scGPT - INFO - match 24285/36601 genes in vocabulary of size 60697.


/usr/local/lib/python3.12/dist-packages/scgpt/model/model.py:77: UserWarning: flash-attn is not installed, using pytorch transformer instead. Set use_fast_transformer=False to avoid this warning. Installing flash-attn is highly recommended.
  warnings.warn(
Embedding cells: 100%|██████████| 821/821 [08:40<00:00,  1.58it/s]
/usr/local/lib/python3.12/dist-packages/legacy_api_wrap/__init__.py:88: FutureWarning: The dtype argument is deprecated and will be removed in late 2024.
  return fn(*args_all, **kw)


Embedding test with fine-tuned scGPT...
scGPT - INFO - match 24285/36601 genes in vocabulary of size 60697.


/usr/local/lib/python3.12/dist-packages/scgpt/model/model.py:77: UserWarning: flash-attn is not installed, using pytorch transformer instead. Set use_fast_transformer=False to avoid this warning. Installing flash-attn is highly recommended.
  warnings.warn(
Embedding cells: 100%|██████████| 291/291 [03:04<00:00,  1.58it/s]
/usr/local/lib/python3.12/dist-packages/legacy_api_wrap/__init__.py:88: FutureWarning: The dtype argument is deprecated and will be removed in late 2024.
  return fn(*args_all, **kw)


Embedding validation with fine-tuned scGPT...
scGPT - INFO - match 2773/3000 genes in vocabulary of size 60697.


/usr/local/lib/python3.12/dist-packages/scgpt/model/model.py:77: UserWarning: flash-attn is not installed, using pytorch transformer instead. Set use_fast_transformer=False to avoid this warning. Installing flash-attn is highly recommended.
  warnings.warn(
Embedding cells: 100%|██████████| 439/439 [03:08<00:00,  2.33it/s]
/usr/local/lib/python3.12/dist-packages/legacy_api_wrap/__init__.py:88: FutureWarning: The dtype argument is deprecated and will be removed in late 2024.
  return fn(*args_all, **kw)


X_train_ft: (52481, 512)
X_test_ft: (18593, 512)
X_val_ft: (28066, 512)


In [29]:
ft_test_results = []

lr_ft = LogisticRegression(
    max_iter=1000,
    C=1.0,
    solver="lbfgs",
    random_state=42,
    n_jobs=-1
)
lr_ft.fit(X_train_ft, y_train)

knn_ft = KNeighborsClassifier(
    n_neighbors=10,
    metric="euclidean",
    n_jobs=-1
)
knn_ft.fit(X_train_ft, y_train)

y_pred_lr_ft_test = lr_ft.predict(X_test_ft)
y_pred_knn_ft_test = knn_ft.predict(X_test_ft)

ft_test_results.append(metric_row("Fine-tuned scGPT + LR", y_test, y_pred_lr_ft_test))
ft_test_results.append(metric_row("Fine-tuned scGPT + KNN k=10", y_test, y_pred_knn_ft_test))

print("Fine-tuned scGPT + LR on original held-out test")
print(classification_report(y_test, y_pred_lr_ft_test, zero_division=0))

print("Fine-tuned scGPT + KNN on original held-out test")
print(classification_report(y_test, y_pred_knn_ft_test, zero_division=0))

adata_val.obs["pred_ft_lr"] = lr_ft.predict(X_val_ft)
adata_val.obs["pred_ft_knn"] = knn_ft.predict(X_val_ft)

ft_test_results_df = pd.DataFrame(ft_test_results)
ft_test_results_df

Fine-tuned scGPT + LR on original held-out test
              precision    recall  f1-score   support

      Acinar       0.96      0.83      0.89      3877
     B-Cells       0.97      1.00      0.98       239
      Ductal       0.89      0.95      0.92      3742
   Endocrine       0.68      0.67      0.68       156
 Endothelial       0.97      0.97      0.97      1454
 Fibroblasts       0.93      0.98      0.95      2361
 Macrophages       0.88      0.99      0.93      1514
        Mast       1.00      0.90      0.95       510
          NK       0.62      0.29      0.39        28
 Neutrophils       1.00      0.96      0.98       181
      Plasma       0.91      0.77      0.84       151
     Schwann       0.70      0.68      0.69        63
    Stellate       0.90      0.92      0.91       462
     T-Cells       0.98      0.99      0.99      3855

    accuracy                           0.94     18593
   macro avg       0.89      0.85      0.86     18593
weighted avg       0.94      0.9

,model,accuracy,balanced_accuracy,precision_macro,recall_macro,f1_macro
0,Fine-tuned scGPT + LR,0.935083,0.850630,0.885433,0.850630,0.862434
1,Fine-tuned scGPT + KNN k=10,0.943904,0.904911,0.909476,0.904911,0.905555


In [30]:
EPOCHS   = 60
LR       = 1e-3
PATIENCE = 10

le_ft = LabelEncoder()
y_train_enc = le_ft.fit_transform(y_train)
y_test_enc = le_ft.transform(y_test)

X_tr = torch.tensor(X_train_ft, dtype=torch.float32)
y_tr = torch.tensor(y_train_enc, dtype=torch.long)
X_te = torch.tensor(X_test_ft, dtype=torch.float32)
y_te = torch.tensor(y_test_enc, dtype=torch.long)

train_dl = DataLoader(
    TensorDataset(X_tr, y_tr),
    batch_size=512,
    shuffle=True,
    pin_memory=(device == "cuda")
)

mlp_ft = MLP(
    in_dim=X_train_ft.shape[1],
    hidden_dims=[256, 128],
    n_classes=len(le_ft.classes_),
    dropout=0.3
).to(device)

optimizer = torch.optim.Adam(mlp_ft.parameters(), lr=LR, weight_decay=1e-4)
criterion = nn.CrossEntropyLoss()
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="max", factor=0.5, patience=5
)

best_val_acc, patience_counter = 0.0, 0
best_ft_path = f"{FT_CACHE}/best_ft_mlp.pt"

for epoch in range(1, EPOCHS + 1):
    mlp_ft.train()
    epoch_loss = 0.0

    for xb, yb in train_dl:
        xb, yb = xb.to(device), yb.to(device)

        optimizer.zero_grad()
        loss = criterion(mlp_ft(xb), yb)
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item() * len(xb)

    mlp_ft.eval()
    with torch.no_grad():
        val_pred = mlp_ft(X_te.to(device)).argmax(1).cpu()
        val_acc = (val_pred == y_te).float().mean().item()

    avg_loss = epoch_loss / len(X_tr)
    scheduler.step(val_acc)

    if epoch % 5 == 0 or epoch == 1:
        print(f"Epoch {epoch:3d}/{EPOCHS}  loss={avg_loss:.4f}  val_acc={val_acc:.4f}")

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        patience_counter = 0
        torch.save(mlp_ft.state_dict(), best_ft_path)
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print(f"Early stopping at epoch {epoch}  (best={best_val_acc:.4f})")
            break

print(f"Best fine-tuned MLP held-out test accuracy: {best_val_acc:.4f}")

mlp_ft.load_state_dict(torch.load(best_ft_path, map_location=device))
mlp_ft.to(device)
mlp_ft.eval()

with torch.no_grad():
    pred_test_enc = mlp_ft(torch.tensor(X_test_ft, dtype=torch.float32).to(device)).argmax(1).cpu().numpy()
    y_pred_mlp_ft_test = le_ft.inverse_transform(pred_test_enc)

ft_test_results.append(metric_row("Fine-tuned scGPT + MLP", y_test, y_pred_mlp_ft_test))

print("Fine-tuned scGPT + MLP on original held-out test")
print(classification_report(y_test, y_pred_mlp_ft_test, zero_division=0))

with torch.no_grad():
    pred_val_enc = mlp_ft(torch.tensor(X_val_ft, dtype=torch.float32).to(device)).argmax(1).cpu().numpy()

adata_val.obs["pred_ft_mlp"] = le_ft.inverse_transform(pred_val_enc)

ft_test_results_df = pd.DataFrame(ft_test_results).sort_values("f1_macro", ascending=False)
ft_test_results_df

Epoch   1/60  loss=0.3411  val_acc=0.9349
Epoch   5/60  loss=0.0791  val_acc=0.9443
Epoch  10/60  loss=0.0660  val_acc=0.9441
Epoch  15/60  loss=0.0615  val_acc=0.9480
Epoch  20/60  loss=0.0595  val_acc=0.9533
Epoch  25/60  loss=0.0574  val_acc=0.9504
Epoch  30/60  loss=0.0555  val_acc=0.9511
Epoch  35/60  loss=0.0544  val_acc=0.9529
Epoch  40/60  loss=0.0533  val_acc=0.9525
Epoch  45/60  loss=0.0463  val_acc=0.9543
Epoch  50/60  loss=0.0468  val_acc=0.9558
Epoch  55/60  loss=0.0463  val_acc=0.9522
Epoch  60/60  loss=0.0418  val_acc=0.9570
Best fine-tuned MLP held-out test accuracy: 0.9570
Fine-tuned scGPT + MLP on original held-out test
              precision    recall  f1-score   support

      Acinar       0.97      0.90      0.94      3877
     B-Cells       0.98      0.99      0.99       239
      Ductal       0.93      0.96      0.94      3742
   Endocrine       0.64      0.83      0.72       156
 Endothelial       0.98      0.97      0.98      1454
 Fibroblasts       0.94      

,model,accuracy,balanced_accuracy,precision_macro,recall_macro,f1_macro
2,Fine-tuned scGPT + MLP,0.957027,0.913361,0.924399,0.913361,0.916645
1,Fine-tuned scGPT + KNN k=10,0.943904,0.904911,0.909476,0.904911,0.905555
0,Fine-tuned scGPT + LR,0.935083,0.850630,0.885433,0.850630,0.862434


In [31]:
ft_val_results = []

pred_cols_ft = {
    "Fine-tuned scGPT + LR": "pred_ft_lr",
    "Fine-tuned scGPT + KNN k=10": "pred_ft_knn",
    "Fine-tuned scGPT + MLP": "pred_ft_mlp",
}

y_true_val = adata_val.obs["cell_type_label"].astype(str).values

for model_name, pred_col in pred_cols_ft.items():
    y_pred_val = adata_val.obs[pred_col].astype(str).values

    ft_val_results.append(metric_row(model_name, y_true_val, y_pred_val))

    print("\n" + "="*80)
    print(model_name, "on validation set")
    print("="*80)
    print(classification_report(y_true_val, y_pred_val, zero_division=0))

ft_val_results_df = pd.DataFrame(ft_val_results).sort_values("f1_macro", ascending=False)
ft_val_results_df

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")



Fine-tuned scGPT + LR on validation set
              precision    recall  f1-score   support

      Acinar       0.99      0.90      0.95      1450
     B-Cells       0.98      0.81      0.89       962
      Ductal       0.96      0.98      0.97      2775
   Endocrine       0.00      0.00      0.00         0
 Endothelial       0.93      0.99      0.96       255
 Fibroblasts       0.91      0.97      0.94       257
 Macrophages       0.33      1.00      0.49      1533
        Mast       0.00      0.00      0.00       225
          NK       0.64      0.88      0.74      1296
 Neutrophils       0.75      0.40      0.53      5054
      Plasma       0.00      0.00      0.00         0
     Schwann       0.00      0.00      0.00         0
    Stellate       1.00      0.81      0.90       534
     T-Cells       0.99      0.93      0.96     13725

    accuracy                           0.83     28066
   macro avg       0.61      0.62      0.59     28066
weighted avg       0.88      0.83      

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")



Fine-tuned scGPT + KNN k=10 on validation set
              precision    recall  f1-score   support

      Acinar       0.99      0.92      0.95      1450
     B-Cells       0.98      0.82      0.89       962
      Ductal       0.96      0.98      0.97      2775
   Endocrine       0.00      0.00      0.00         0
 Endothelial       0.91      1.00      0.96       255
 Fibroblasts       0.89      0.96      0.92       257
 Macrophages       0.33      0.99      0.49      1533
        Mast       0.26      0.04      0.06       225
          NK       0.60      0.89      0.71      1296
 Neutrophils       0.29      0.41      0.34      5054
      Plasma       0.00      0.00      0.00         0
     Schwann       0.00      0.00      0.00         0
    Stellate       1.00      0.83      0.91       534
     T-Cells       0.99      0.59      0.74     13725

    accuracy                           0.66     28066
   macro avg       0.59      0.60      0.57     28066
weighted avg       0.80      0.66

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")



Fine-tuned scGPT + MLP on validation set
              precision    recall  f1-score   support

      Acinar       0.98      0.88      0.93      1450
     B-Cells       0.99      0.80      0.89       962
      Ductal       0.96      0.97      0.97      2775
   Endocrine       0.00      0.00      0.00         0
 Endothelial       0.89      0.99      0.94       255
 Fibroblasts       0.60      0.93      0.73       257
 Macrophages       0.33      0.98      0.50      1533
        Mast       0.00      0.00      0.00       225
          NK       0.82      0.33      0.47      1296
 Neutrophils       0.16      0.43      0.23      5054
      Plasma       0.00      0.00      0.00         0
     Schwann       0.00      0.00      0.00         0
    Stellate       1.00      0.53      0.69       534
     T-Cells       0.99      0.21      0.34     13725

    accuracy                           0.45     28066
   macro avg       0.55      0.50      0.48     28066
weighted avg       0.78      0.45     

,model,accuracy,balanced_accuracy,precision_macro,recall_macro,f1_macro
0,Fine-tuned scGPT + LR,0.825340,0.788849,0.605893,0.61981,0.593883
1,Fine-tuned scGPT + KNN k=10,0.664576,0.764947,0.586199,0.60103,0.567948
2,Fine-tuned scGPT + MLP,0.445379,0.641938,0.552369,0.50438,0.477576


In [32]:
OUT_H5AD_FT = f"{VAL_DIR}/validation_with_finetuned_scgpt_predictions_FIXED.h5ad"
OUT_OBS_FT = f"{VAL_DIR}/validation_finetuned_scgpt_predictions_FIXED_obs.csv"
OUT_VAL_METRICS_FT = f"{VAL_DIR}/validation_finetuned_scgpt_metrics_FIXED.csv"
OUT_TEST_METRICS_FT = f"{VAL_DIR}/heldout_test_finetuned_scgpt_metrics_FIXED.csv"

adata_val.write_h5ad(OUT_H5AD_FT)
adata_val.obs.to_csv(OUT_OBS_FT)
ft_val_results_df.to_csv(OUT_VAL_METRICS_FT, index=False)
ft_test_results_df.to_csv(OUT_TEST_METRICS_FT, index=False)

print("Saved:", OUT_H5AD_FT)
print("Saved:", OUT_OBS_FT)
print("Saved:", OUT_VAL_METRICS_FT)
print("Saved:", OUT_TEST_METRICS_FT)

adata_val.obs[
    ["cell_type_label", "pred_ft_lr", "pred_ft_knn", "pred_ft_mlp"]
].head()

Saved: /content/drive/MyDrive/CompGenomicsValidation/validation_with_finetuned_scgpt_predictions_FIXED.h5ad
Saved: /content/drive/MyDrive/CompGenomicsValidation/validation_finetuned_scgpt_predictions_FIXED_obs.csv
Saved: /content/drive/MyDrive/CompGenomicsValidation/validation_finetuned_scgpt_metrics_FIXED.csv
Saved: /content/drive/MyDrive/CompGenomicsValidation/heldout_test_finetuned_scgpt_metrics_FIXED.csv


,cell_type_label,pred_ft_lr,pred_ft_knn,pred_ft_mlp
AAACCCAGTCCGACGT-1_AdjNorm_TISSUE_1,Stellate,Stellate,Stellate,Fibroblasts
AAACGAAAGCGACCCT-1_AdjNorm_TISSUE_1,Acinar,Acinar,Acinar,Acinar
AAACGAAAGCGGACAT-1_AdjNorm_TISSUE_1,T-Cells,T-Cells,T-Cells,Neutrophils
AAACGAAAGTCACACT-1_AdjNorm_TISSUE_1,Acinar,Acinar,Acinar,Acinar
AAACGAACAACAGATA-1_AdjNorm_TISSUE_1,Macrophages,Macrophages,Macrophages,Macrophages
